# Simple BPE Tokenizer
A trained BPE takes an input text, splits it up into tokens, and assigns a token ID (not to be confused with the numerical vector embeddings of said token):

```{figure} ../../figures/class1/004_BPE.png
---
name: BPE-high-level
---
Modified from [Sebastian Rasckha](https://github.com/rasbt/LLMs-from-scratch/blob/main) under the [APACHE License](https://github.com/rasbt/LLMs-from-scratch/blob/main/LICENSE.txt).

This is what we'll aim to do!

<style>

.footnote-list {
    display: none;
}
</style>

## 4.1 Training
For **Training BPE**, we begin with a vocabulary that is a set of all individual characters (& optionally some "special character tokens", we won't do this here. See [Wiki/BPE](https://en.wikipedia.org/wiki/Byte-pair_encoding#Modified_algorithm))

Then we do these these steps:
1. **Count** the most frequent pairs of characters *or* bytes[^bytes_ex] in our corpus 
2. **Merge** that pair 
3. **Replace** that pair with a new "token" encoding
3. **Repeat** 1 & 2 until there are no gains 
    - We've hit "max" vocabulary length, set by a parameter *vocab_size* or *k* in [Jurafsky & Martin](https://web.stanford.edu/~jurafsky/slp3/2.pdf).


[^bytes_ex]: : LLMs use Byte-level BPE, not character-level. We'll start with words for intution, then transition to bytes.

### 1. Count Frequent Pairs
Let's start with an example of a simple corpus consisting of five unique words:
```
"hug", "pug", "pun", "bun", "hugs"
```

For this example, the base vocabulary is:
```
["b", "g", "h", "n", "p", "s", "u"]
```

Let's say that our corpus has these frequency counts:
```
("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)
```

We begin training by splitting each word into characters (our base vocabulary)
```
("h" "u" "g", 10), ("p" "u" "g", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "u" "g" "s", 5)
```

Then we look at pairs of characters. Can we find any character pairs that are often represented together? Seems "Hug" and "Hugs" is represented 15 times in total, so a guess could be "hu"? But it actually isn't ! 

:::{admonition} QUESTION: 
:class: red
Can you find the most frequent pair of characters?

<details>
<summary>Click to see ANSWER</summary>
"Hug" and "Hugs" are good places to start.
<br><br>
There is actually another word that shares characters with "hug" and "hugs" which is "pug", so the most frequent pair is <strong>ug</strong>.
</details>
:::

:::{admonition} NOTE. JURAFSKY & MARTIN EXAMPLE
:class: blue, dropdown
In this week's reading from [Jurafsky & Martin (p. 11)](https://web.stanford.edu/~jurafsky/slp3/2.pdf), the above example
```
h u g p u g p u n b u n h u g s 
```

Is basically equivalent to 
```
A B D C A B E D C A B
```

Except that we're techically representing each character with an added frequency 
```
("h" "u" "g", 10) -> "h u g h u g h u g h u g h u g h u g h u g ...." (10 times)
```

::::


In [42]:
text = "hug hug hug hug hug hug hug hug hug hug pug pug pug pug pug pun pun pun pun pun pun pun pun pun pun pun pun bun bun bun bun hugs hugs hugs hugs hugs"
#tokens = list(text.encode("utf-8"))

def get_stats(ids, counts=None):
    """
    Given a list of integers, return a dictionary of counts of consecutive pairs
    Example: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    Optionally allows to update an existing dictionary of counts
    """
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]): # iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [43]:
stats = get_stats(text)
print(f"Total number of unique pairs: {len(stats)}")

# Show top 10 most frequent pairs
top_pairs = sorted([(count, pair) for pair, count in stats.items()], reverse=True)[:10]
print("\nTop 10 most frequent pairs:")
for count, pair in top_pairs:
    print(f"  {pair}: {count} times")

Total number of unique pairs: 12

Top 10 most frequent pairs:
  ('u', 'g'): 20 times
  ('p', 'u'): 17 times
  (' ', 'p'): 17 times
  ('u', 'n'): 16 times
  ('n', ' '): 16 times
  ('h', 'u'): 15 times
  ('g', ' '): 15 times
  (' ', 'h'): 14 times
  ('g', 's'): 5 times
  ('s', ' '): 4 times


### 1. Count **Encoded** Pairs: Unicode, Bits & Bytes
The example above used *words*, but in reality we're using Unicode, a **character set** where all characters have number id (**Code Points**). For Hello this would be:

```
U+0068 U+0065 U+006C U+006C U+006F
```
Unicode has +1500000 **code points**. This is not computionally efficient. Luckily, we can convert this into bytes with the **UTF-8 encoding**:

In [29]:
text = "hello"
encoding = list(text.encode("utf-8"))
print(encoding)

[104, 101, 108, 108, 111]


Note that this is represented in decimal numbers, but that Jurafsky & Martin explain it with hexidecimals: 
```python
decimal    hexadecimal
104        68
101        65
108        6C
108        6C
111        6F
```

In [30]:
def get_stats(ids, counts=None):
    """
    Given a list of integers, return a dictionary of counts of consecutive pairs
    Example: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    Optionally allows to update an existing dictionary of counts
    """
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]): # iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts

## References
This NB is partially inspired by Sebastian Rasckha's "Build a Large Language Model from Scratch" and HF's [BPE tutorial](https://huggingface.co/learn/llm-course/en/chapter6/5), both licensed under APACHE.

``